# 00 — Euclidean Distance (The Wrong but Useful Way)

The distance formula from algebra class works perfectly in two-dimensional Cartesian space. Latitude and longitude look like a two-dimensional Cartesian space. This notebook is about what happens when you apply the former to the latter — and why the result is wrong, how wrong it is, and when you can get away with it anyway.

## 1. The Distance Formula

Given two points in a flat plane, the straight-line distance between them is:

```text
d = √( (x₂ - x₁)² + (y₂ - y₁)² )
```

This is the Pythagorean theorem applied to the right triangle formed by the two points and their shared corner. It works because the axes are in consistent, uniform units — both in meters, both in pixels, both in whatever.

Mapping this onto coordinates:

```text
x  →  longitude
y  →  latitude
```

The formula becomes:

```python
import math
d = math.sqrt((lon2 - lon1)**2 + (lat2 - lat1)**2)
```

And you get a number. The question — which this notebook will make you earn — is: **a number in what units?**

## 2. Compute Distance Between Two Points

In [ ]:
import math

# Two points in [lon, lat] order — GeoJSON convention
p1 = (-98.5, 33.8)   # near Wichita Falls, TX
p2 = (-97.2, 34.1)   # about 130 km to the northeast

dx = p2[0] - p1[0]   # longitude difference
dy = p2[1] - p1[1]   # latitude difference

d = math.sqrt(dx**2 + dy**2)

print(f"p1: {p1}")
print(f"p2: {p2}")
print(f"dx (Δ lon): {dx}")
print(f"dy (Δ lat): {dy}")
print(f"d:  {d:.4f}")

## 3. What Are the Units?

You just computed `d ≈ 1.334`.

**1.334 what?**

Not kilometers. Not miles. Not meters.

**Degrees.**

`dx` and `dy` are differences in degrees of longitude and latitude. Squaring them, adding them, and taking the square root gives you a number in degrees — a hybrid angular unit that combines east-west and north-south angle into a single diagonal measurement.

That is not a physically meaningful distance. A degree of longitude at 33°N covers about 93 km. A degree of latitude covers about 111 km. They are different lengths. Treating them as equivalent axes — which is exactly what the Euclidean formula does — introduces an immediate geometric error.

The actual distance between these two points is roughly **133 km**. The Euclidean formula gave `1.334` — which is off by a factor of ~100, and the factor depends entirely on where on Earth you are.

In [ ]:
# Wrap it as a reusable function
def euclidean_distance(p1, p2):
    """
    Straight-line distance between two [lon, lat] points.
    Returns a value in degrees — NOT kilometers.
    Use only for relative comparisons within small, consistent areas.
    """
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.sqrt(dx**2 + dy**2)


print(f"Euclidean distance: {euclidean_distance(p1, p2):.4f}°")
print("(Actual distance is ~133 km — the degree value is not comparable to km)")

## 4. Visualize on a Map

Plotting both points with a line between them makes the geometry concrete. It also makes it obvious that the "distance" is a diagonal across the coordinate grid — not a physical path across the ground.

In [ ]:
from ipyleaflet import Map, GeoJSON

points_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p1)}, "properties": {"name": "p1"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p2)}, "properties": {"name": "p2"}},
    ]
}

line_fc = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [list(p1), list(p2)]},
        "properties": {"label": f"d = {euclidean_distance(p1, p2):.4f}°  (not km)"}
    }]
}

center_lat = (p1[1] + p2[1]) / 2
center_lon = (p1[0] + p2[0]) / 2

m = Map(center=(center_lat, center_lon), zoom=9)
m.add(GeoJSON(data=points_fc))
m.add(GeoJSON(data=line_fc, style={"color": "#e63946", "weight": 2}))
m

## 5. When This Works

Euclidean distance on raw degrees is not entirely useless. Within a small, geographically consistent area, the degree-to-km ratio is approximately constant across all the points in your dataset — so the *relative ordering* of distances is roughly preserved even if the *absolute values* are wrong.

**Acceptable use cases:**
- Finding the nearest point out of a local set (same city, same county)
- Rough filtering — "which features are in the same neighborhood?"
- Sorting by proximity when you only care about rank, not exact distance
- Quick checks during development before switching to proper formulas

**Not acceptable:**
- Reporting an actual distance to a user
- Comparing distances across different latitudes
- Any computation where someone acts on the result
- Navigation, guidance, or range estimation of any kind

The test: *does the answer matter?* If yes, use Haversine. If you just need "roughly which direction is closer," Euclidean on degrees can hold the line temporarily.

---

## Exercise A — Distances Between Several Points

Compute the Euclidean distance from the base point to each of the targets below. Print the results sorted nearest to farthest.

In [ ]:
base = (-98.47, 33.91)

targets = [
    ('Tinker AFB',         (-97.37, 35.39)),
    ('NAS Fort Worth JRB', (-97.04, 32.85)),
    ('Lubbock',            (-101.87, 33.57)),
    ('Oklahoma City',      (-97.52, 35.47)),
    ('Abilene',            (-99.73, 32.45)),
]

rows = sorted((euclidean_distance(base, coords), name) for name, coords in targets)
for dist, name in rows:
    print(f'{name:<22} {dist:.4f}°')


## Exercise B — Rank Nearest Points

Using `euclidean_distance`, write a function `nearest_n(base, points, n)` that returns the `n` closest points from a list, sorted by distance. Test it on the targets above.

In [ ]:
def nearest_n(base, named_points, n):
    """
    Returns the n closest (name, coord) pairs from named_points to base,
    sorted nearest first.
    named_points: list of (name, (lon, lat))
    """
    return sorted(named_points, key=lambda item: euclidean_distance(base, item[1]))[:n]


top3 = nearest_n(base, targets, 3)
for name, coord in top3:
    print(f"{name}: {euclidean_distance(base, coord):.4f}°")


## Exercise C — Visualize All Distances

Plot `base` and all five targets on a map. Draw a line from `base` to each target. Label each line with the Euclidean distance value (in the `properties` dict — you can inspect it in the browser dev tools or just print it before displaying).

In [ ]:
from ipyleaflet import Map, GeoJSON

features = [
    {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': list(base)}, 'properties': {'name': 'Base'}}
]
line_features = []
for name, coords in targets:
    features.append({'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': list(coords)}, 'properties': {'name': name}})
    line_features.append({
        'type': 'Feature',
        'geometry': {'type': 'LineString', 'coordinates': [list(base), list(coords)]},
        'properties': {'name': name, 'euclidean_deg': round(euclidean_distance(base, coords), 4)},
    })

print('Line properties:')
for line in line_features:
    print(line['properties'])

m = Map(center=(base[1], base[0]), zoom=6)
m.add(GeoJSON(data={'type': 'FeatureCollection', 'features': features}))
m.add(GeoJSON(data={'type': 'FeatureCollection', 'features': line_features}, style={'color': '#e63946', 'weight': 2}))
m


No. One degree north-south is roughly 111 km, while one degree east-west at Texas latitudes is closer to the low 90 km range because longitude lines converge toward the poles. So the northward target is physically farther even though both look like `1.0000°` in raw Euclidean degree space.


## Next

In [01 — Haversine Distance](./01-Haversine_Distance.ipynb), we replace the Euclidean formula with one that accounts for the curvature of the Earth — producing real distances in kilometers that hold up anywhere on the globe.